# Continue a finished run for ~2 h (fresh short LR schedule from its weights) → submission.zip

**Inputs:** the finished run's notebook *output* (`model.py`, `run_config.json`, `run/<run>_step*.pt` / `_best.pt`) **and** `prefixlm-run1`.
Settings: GPU T4 ×2, Internet On, secret `HF_TOKEN`. Preset for arch 4: nothing to change, just *Save & Run All*. It continues arch 4 at its original final learning rate (3e-5, constant) with its optimizer state, i.e. the same run extended.

Everything except the LR schedule is taken unchanged from the source run's `run_config.json` (same data, mix, tokens/step, validation windows).
At the end the continued model's best validation loss is compared with the source run's; if it is not lower, no zip is written and your existing submission stands.

In [ ]:
# 1 — setup + clock
import os, json, glob, shutil, time, copy, datetime as dt
SESSION_START = time.time()
!pip install -q bert-score
import torch
NGPU = torch.cuda.device_count(); print("GPUs:", NGPU)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret not available:", e)
os.chdir("/kaggle/working")

In [ ]:
# 2 — CONFIG
TODO = "TODO"
SRC_RUN = "arch4"
FINISH_BY_IST = "23:20"
NEW_SCHEDULE = dict(
    lr_max=3e-5, lr_min=3e-5, warmup_steps=0, schedule="constant",
    decay_frac=None,
)
INIT_OPTIMIZER = True
CANDS = {
    "greedy_nr4":       dict(temperature=0, top_k=0, top_p=1.0, rep_penalty=1.0, no_repeat_ngram=4),
    "greedy_rp1.2":     dict(temperature=0, top_k=0, top_p=1.0, rep_penalty=1.2, no_repeat_ngram=0),
    "greedy_nr3_rp1.1": dict(temperature=0, top_k=0, top_p=1.0, rep_penalty=1.1, no_repeat_ngram=3),
}
RESERVE_MIN = 10 + 3 * len(CANDS)

missing = [k for k, v in NEW_SCHEDULE.items() if v == TODO]
assert not missing, f"fill NEW_SCHEDULE: {missing}"

def find(pattern):
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

steps = sorted(find(f"{SRC_RUN}_step*.pt"), key=lambda q: int(q.split("_step")[-1][:-3]))
best = find(f"{SRC_RUN}_best.pt")
INIT = steps[-1] if steps else (best[0] if best else None)
assert INIT, f"no {SRC_RUN}_step*.pt or {SRC_RUN}_best.pt under /kaggle/input — attach the source run's output"
root = os.path.dirname(os.path.dirname(INIT)) if os.path.basename(os.path.dirname(INIT)) == "run" else os.path.dirname(INIT)
def near(fn):
    hits = sorted(glob.glob(f"{root}/**/{fn}", recursive=True), key=len)
    assert hits, f"{fn} not found next to {INIT}"
    return hits[0]
shutil.copy(near("model.py"), "model.py")
CFG = json.load(open(near("run_config.json")))
assert CFG["run"]["run_name"] == SRC_RUN, CFG["run"]["run_name"]
SRC_VAL = None
if best:
    SRC_VAL = torch.load(best[0], map_location="cpu", weights_only=False).get("val_loss")
metas = [m for m in find("meta.json") if glob.glob(os.path.join(os.path.dirname(m), "val_*.bin"))]
assert metas, "attach prefixlm-run1"
TOKEN_DIR = os.path.dirname(metas[0])
os.makedirs("tokenizer", exist_ok=True)
for fn in ["vocab.json", "merges.txt"]:
    shutil.copy(find(fn)[0], f"tokenizer/{fn}")

IST = dt.timezone(dt.timedelta(hours=5, minutes=30))
_now = dt.datetime.now(IST); _h, _m = map(int, FINISH_BY_IST.split(":"))
_fin = _now.replace(hour=_h, minute=_m, second=0, microsecond=0)
SESSION_HOURS = (_fin.timestamp() - SESSION_START) / 3600
assert SESSION_HOURS * 60 > RESERVE_MIN + 30, f"only {SESSION_HOURS*60:.0f} min until {FINISH_BY_IST}: not worth continuing"

NEW = f"{SRC_RUN}_cont"
CFG["run"].update(run_name=NEW, session_start=SESSION_START, session_hours=SESSION_HOURS,
                  reserve_minutes_after_train=RESERVE_MIN, token_dir=TOKEN_DIR,
                  tokenizer_dir="/kaggle/working/tokenizer", out_dir="/kaggle/working/run", resume_dirs=[],
                  smoke_test=False, init_from=INIT, init_optimizer=INIT_OPTIMIZER, eval_ckpt_overhead_frac=0.06)
CFG["train"].update(NEW_SCHEDULE, max_tokens=None, val_every_min=15, ckpt_every_min=20)
CFG["gen"]["checkpoint"] = "best"
json.dump(CFG, open("run_config.json", "w"), indent=1)
print(f"continue {SRC_RUN} from {INIT}\n  source best val: {SRC_VAL}\n  training until "
      f"{dt.datetime.fromtimestamp(SESSION_START + SESSION_HOURS*3600 - RESERVE_MIN*60, IST):%H:%M} IST, zip by {FINISH_BY_IST}")
print("  new schedule:", NEW_SCHEDULE)
from huggingface_hub import hf_hub_download
TEST_PATH = hf_hub_download("Exploration-Lab/CS-781-Capstone", "Phase_1_Pretraining/Test/test.json", repo_type="dataset")

In [ ]:
%%writefile train.py
import contextlib, glob, json, math, os, sys, time

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP

CFG = json.load(open(sys.argv[1]))
RUN, MODEL_CFG, TR = CFG["run"], CFG["model"], CFG["train"]

if RUN.get("smoke_test"):
    import dummy_model as model_lib
else:
    import model as model_lib


# ----------------------------------------------------------------------------- setup
def setup_dist():
    if int(os.environ.get("WORLD_SIZE", "1")) > 1:
        dist.init_process_group("nccl")
        rank, world, local = dist.get_rank(), dist.get_world_size(), int(os.environ["LOCAL_RANK"])
    else:
        rank, world, local = 0, 1, 0
    torch.cuda.set_device(local)
    return rank, world, torch.device("cuda", local)


RANK, WORLD, DEV = setup_dist()
IS_MAIN = RANK == 0
DDP_ON = WORLD > 1


def log(*a):
    if IS_MAIN:
        print(*a, flush=True)


def bcast_float(x):
    t = torch.tensor([float(x)], device=DEV)
    if DDP_ON:
        dist.broadcast(t, 0)
    return t.item()


def allreduce_sum(*vals):
    t = torch.tensor([float(v) for v in vals], device=DEV, dtype=torch.float64)
    if DDP_ON:
        dist.all_reduce(t)
    return t.tolist()


torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

# ----------------------------------------------------------------------------- time budget
SESSION_START = RUN["session_start"]
SMOKE = bool(RUN.get("smoke_test"))
if SMOKE:
    TRAIN_DEADLINE = time.time() + RUN.get("smoke_train_minutes", 3) * 60
else:
    TRAIN_DEADLINE = (SESSION_START + RUN["session_hours"] * 3600
                      - RUN["reserve_minutes_after_train"] * 60)
log(f"Train deadline in {(TRAIN_DEADLINE - time.time())/3600:.2f} h")

# ----------------------------------------------------------------------------- data
META = json.load(open(os.path.join(RUN["token_dir"], "meta.json")))
CTX = int(TR["ctx_len"])


class WindowSampler:

    def __init__(self, split, mix, seed):
        files = sorted(glob.glob(os.path.join(RUN["token_dir"], f"{split}_*.bin")))
        assert files, f"no {split}_*.bin files in {RUN['token_dir']}"
        self.names, self.arrs = [], []
        for f in files:
            a = np.memmap(f, dtype=np.uint16, mode="r")
            if a.size > CTX + 2:
                self.names.append(os.path.basename(f)[len(split) + 1:-4])
                self.arrs.append(a)
        if mix == "proportional":
            w = np.array([a.size for a in self.arrs], dtype=np.float64)
        else:
            w = np.array([float(mix.get(n, 0.0)) for n in self.names])
        assert w.sum() > 0, f"mix gives zero weight to all of {self.names}"
        self.w = w / w.sum()
        self.rng = np.random.default_rng(seed)

    def window(self, i):
        a = self.arrs[i]
        s = int(self.rng.integers(0, a.size - CTX - 1))
        return np.asarray(a[s:s + CTX + 1], dtype=np.int64)

    def batch(self, B):
        idx = self.rng.choice(len(self.arrs), size=B, p=self.w)
        return torch.from_numpy(np.stack([self.window(i) for i in idx]))


def prefix_lengths(B, mode, fmin, fmax, rng):
    if mode == "none":
        return np.zeros(B, dtype=np.int64)
    lo = int(round(fmin * CTX))
    if mode == "fixed":
        return np.full(B, lo, dtype=np.int64)
    if mode == "random":
        hi = int(round(fmax * CTX))
        return rng.integers(lo, hi + 1, size=B).astype(np.int64)
    if mode == "batch_random":
        hi = int(round(fmax * CTX))
        return np.full(B, int(rng.integers(lo, hi + 1)), dtype=np.int64)
    raise ValueError(f"prefix_mode must be none|fixed|random|batch_random, got {mode!r}")


def make_xy(chunk, plen):
    x, y = chunk[:, :-1].clone(), chunk[:, 1:].clone()
    pos = torch.arange(CTX)[None, :]
    y[pos < (torch.from_numpy(plen)[:, None] - 1)] = -100
    return x, y, torch.from_numpy(plen)


train_sampler = WindowSampler("train", TR["mix"], seed=1234 + RANK * 7919)
train_prefix_rng = np.random.default_rng(99 + RANK)
log(f"Sources: {dict(zip(train_sampler.names, np.round(train_sampler.w, 3)))}")


VAL_PER_SOURCE = 8 if SMOKE else int(RUN.get("val_windows_per_source", 48))
val_sampler = WindowSampler("val", "proportional", seed=4321)
VAL = []
vp_rng = np.random.default_rng(7)
for si, name in enumerate(val_sampler.names):
    for _ in range(VAL_PER_SOURCE):
        c = torch.from_numpy(val_sampler.window(si))[None]
        pl = prefix_lengths(1, TR["val_prefix_mode"], TR["val_prefix_frac"], TR["val_prefix_frac"], vp_rng)
        x, y, p = make_xy(c, pl)
        VAL.append((name, x, y, p))
VAL_SHARD = VAL[RANK::WORLD]

# ----------------------------------------------------------------------------- model
MODEL_CFG.setdefault("vocab_size", META["vocab_size"])
MODEL_CFG.setdefault("max_seq_len", CTX)
raw_model = model_lib.build_model(MODEL_CFG).to(DEV)
n_params = sum(p.numel() for p in raw_model.parameters())
log(f"Parameters: {n_params/1e6:.2f}M")
if not SMOKE:
    assert 50e6 <= n_params <= 300e6, "Phase-1 rule: model must be 50M-300M parameters"
if TR.get("grad_ckpt") and hasattr(raw_model, "set_gradient_checkpointing"):
    raw_model.set_gradient_checkpointing(True)

decay, no_decay = [], []
for n, p in raw_model.named_parameters():
    if p.requires_grad:
        (decay if (p.dim() >= 2 or not TR["wd_exclude_1d"]) else no_decay).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": TR["weight_decay"]},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=TR["lr_max"], betas=(TR["beta1"], TR["beta2"]), eps=1e-8, fused=True)
scaler = torch.amp.GradScaler("cuda", enabled=TR.get("amp_dtype", "fp16") == "fp16")
AMP_DTYPE = torch.float16 if TR.get("amp_dtype", "fp16") == "fp16" else torch.bfloat16

model = raw_model
if TR.get("compile"):
    model = torch.compile(model)
if DDP_ON:
    model = DDP(model, device_ids=[DEV.index], find_unused_parameters=bool(TR.get("find_unused", False)))


def unwrap():
    return raw_model


# ----------------------------------------------------------------------------- checkpoints
OUT = RUN["out_dir"]
os.makedirs(OUT, exist_ok=True)
NAME = RUN["run_name"]


def ckpt_files(d):
    return sorted(glob.glob(os.path.join(d, f"{NAME}_step*.pt")))


def save_ckpt(state, path):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)


def save_training_ckpt(step, tokens, total_steps, best_val):
    if not IS_MAIN:
        return
    path = os.path.join(OUT, f"{NAME}_step{step:07d}.pt")
    save_ckpt(dict(model=unwrap().state_dict(), optimizer=optimizer.state_dict(),
                   scaler=scaler.state_dict(), step=step, tokens=tokens,
                   total_steps=total_steps, best_val=best_val, cfg=CFG), path)
    for old in ckpt_files(OUT)[:-2]:
        os.remove(old)
    log(f"  saved {os.path.basename(path)}")


def save_model_only(tag, step, tokens, val):
    if IS_MAIN:
        save_ckpt(dict(model=unwrap().state_dict(), step=step, tokens=tokens, val_loss=val,
                       model_cfg=MODEL_CFG, n_params=n_params),
                  os.path.join(OUT, f"{NAME}_{tag}.pt"))


step, tokens, total_steps, best_val = 0, 0, None, float("inf")
RESUMED = False
cands = []
for d in [OUT] + list(RUN.get("resume_dirs", [])):
    cands += ckpt_files(d)
if cands:
    latest = max(cands, key=lambda p: int(p.split("_step")[-1][:-3]))
    ck = torch.load(latest, map_location=DEV, weights_only=False)
    unwrap().load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scaler.load_state_dict(ck["scaler"])
    step, tokens, total_steps, best_val = ck["step"], ck["tokens"], ck["total_steps"], ck["best_val"]
    log(f"Resumed from {latest} (step {step}, {tokens/1e6:.1f}M tokens)")
    RESUMED = True
    del ck

    train_sampler.rng = np.random.default_rng(1234 + RANK * 7919 + 1_000_003 * step)
    train_prefix_rng = np.random.default_rng(99 + RANK + 1_000_003 * step)
    log("Data sampler re-seeded for resume")


INIT_FROM = RUN.get("init_from")
if INIT_FROM and not RESUMED:
    ck = torch.load(INIT_FROM, map_location=DEV, weights_only=False)
    unwrap().load_state_dict(ck["model"])
    if RUN.get("init_optimizer") and "optimizer" in ck:
        optimizer.load_state_dict(ck["optimizer"])
    log(f"Initialised from {INIT_FROM} (its step {ck.get('step')}, {ck.get('tokens', 0)/1e6:.0f}M tokens) | "
        f"optimizer state {'loaded' if RUN.get('init_optimizer') and 'optimizer' in ck else 'fresh'} | new schedule")

    train_sampler.rng = np.random.default_rng(1234 + RANK * 7919 + 7_777_777 + int(ck.get("step") or 0))
    train_prefix_rng = np.random.default_rng(99 + RANK + 7_777_777 + int(ck.get("step") or 0))
    del ck

# ----------------------------------------------------------------------------- schedule
def lr_at(s, total):
    lr_max, lr_min, warm = TR["lr_max"], TR["lr_min"], TR["warmup_steps"]
    if s < warm:
        return lr_max * (s + 1) / warm
    if total is None:
        return lr_max
    t = min(1.0, (s - warm) / max(1, total - warm))
    kind = TR["schedule"]
    if kind == "cosine":
        return lr_min + (lr_max - lr_min) * 0.5 * (1 + math.cos(math.pi * t))
    if kind == "linear":
        return lr_max + (lr_min - lr_max) * t
    if kind == "constant":
        return lr_max
    if kind == "wsd":
        d = TR["decay_frac"]
        return lr_max if t < 1 - d else lr_max + (lr_min - lr_max) * (t - (1 - d)) / d
    raise ValueError(f"unknown schedule {kind!r}")


def align_targets(logits, y):
    L, T = logits.size(1), y.size(1)
    if L == T:
        return y
    if L > T or bool((y[:, :T - L] != -100).any()):
        raise ValueError(f"model returned {L} logit positions for {T} targets, but some dropped "
                         f"targets are scored: return at least T - prefix_len + 1 positions")
    return y[:, T - L:]


# ----------------------------------------------------------------------------- eval
@torch.no_grad()
def evaluate():
    model.eval()
    sums = {}
    for name, x, y, p in VAL_SHARD:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = unwrap()(x.to(DEV), p.to(DEV))
        y = align_targets(logits, y.to(DEV))
        l = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), y.reshape(-1),
                            ignore_index=-100, reduction="sum").item()
        n = (y != -100).sum().item()
        s = sums.setdefault(name, [0.0, 0])
        s[0] += l; s[1] += n
    names = val_sampler.names
    flat = []
    for n in names:
        flat += sums.get(n, [0.0, 0])
    flat = allreduce_sum(*flat)
    per = {n: flat[2 * i] / max(1, flat[2 * i + 1]) for i, n in enumerate(names)}
    tot = sum(flat[0::2]) / max(1, sum(flat[1::2]))
    model.train()
    return tot, per


# ----------------------------------------------------------------------------- train loop
MB, ACC = int(TR["micro_batch"]), int(TR["grad_accum"])
TOK_PER_STEP = MB * CTX * ACC * WORLD
log(f"Tokens/step: {TOK_PER_STEP:,}  (micro {MB} x ctx {CTX} x accum {ACC} x gpus {WORLD})")

csv_path = os.path.join(OUT, f"{NAME}_log.csv")
if IS_MAIN and not os.path.exists(csv_path):
    open(csv_path, "w").write("step,tokens,train_loss,lr,tok_per_s,elapsed_h,val_loss,val_per_source,"
                               "grad_norm_max,amp_scale,skipped_steps\n")


def csv(row):
    if IS_MAIN:
        open(csv_path, "a").write(row + "\n")


CALIB_AFTER = 10
CALIB_STEPS = 30
calib_t0 = None
run_t0 = time.time()
last_ckpt = time.time()
last_val = time.time()
win_t0, win_tok = time.time(), 0
model.train()
optimizer.zero_grad(set_to_none=True)
steps_this_run = 0


gn_max_win = torch.zeros((), device=DEV)
skipped_total, skipped_win = 0, 0

while True:

    loss_acc = torch.zeros((), device=DEV)
    for micro in range(ACC):
        chunk = train_sampler.batch(MB)
        pl = prefix_lengths(MB, TR["prefix_mode"], TR.get("prefix_frac_min", 0.0),
                            TR.get("prefix_frac_max", 0.0), train_prefix_rng)
        x, y, p = make_xy(chunk, pl)
        x, y, p = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True), p.to(DEV, non_blocking=True)
        sync_ctx = model.no_sync() if (DDP_ON and micro < ACC - 1) else contextlib.nullcontext()
        with sync_ctx:
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(x, p)
                yy = align_targets(logits, y)
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)).float(), yy.reshape(-1),
                                       ignore_index=-100)
            scaler.scale(loss / ACC).backward()
        loss_acc += loss.detach() / ACC

    lr = lr_at(step, total_steps)
    for g in optimizer.param_groups:
        g["lr"] = lr
    scaler.unscale_(optimizer)
    gn = torch.nn.utils.clip_grad_norm_(raw_model.parameters(), TR["grad_clip"])
    gn_max_win = torch.maximum(gn_max_win, torch.nan_to_num(gn.detach().float(), nan=float("inf")))
    scale_before = scaler.get_scale()
    scaler.step(optimizer)
    scaler.update()
    if scaler.is_enabled() and scaler.get_scale() < scale_before:
        skipped_total += 1; skipped_win += 1
    optimizer.zero_grad(set_to_none=True)
    step += 1
    steps_this_run += 1
    tokens += TOK_PER_STEP
    win_tok += TOK_PER_STEP


    if steps_this_run == CALIB_AFTER:
        torch.cuda.synchronize(); calib_t0 = time.time()
    if steps_this_run == CALIB_AFTER + CALIB_STEPS and (total_steps is None or RESUMED):
        torch.cuda.synchronize()
        sec_per_step = (time.time() - calib_t0) / CALIB_STEPS
        overhead = 1.0 - RUN.get("eval_ckpt_overhead_frac", 0.04)
        est = step + int((TRAIN_DEADLINE - time.time()) / sec_per_step * overhead)
        if TR.get("max_tokens"):
            est = min(est, int(TR["max_tokens"]) // TOK_PER_STEP)
        est = int(bcast_float(est))
        if RESUMED and total_steps is not None:
            if est < total_steps:
                log(f"Resume: only time for {est} of {total_steps} scheduled steps -> schedule shortened so LR still decays")
                total_steps = est
        else:
            total_steps = est
        log(f"Measured {sec_per_step:.2f}s/step ({TOK_PER_STEP/sec_per_step/1e3:.1f}K tok/s). "
            f"Schedule sized to {total_steps} steps = {total_steps*TOK_PER_STEP/1e6:.0f}M tokens.")

    now = time.time()
    if step % TR.get("log_every_steps", 20) == 0:
        l = allreduce_sum(loss_acc.item())[0] / WORLD
        tps = win_tok / max(1e-9, now - win_t0)
        gmax = gn_max_win.item()
        sc = scaler.get_scale() if scaler.is_enabled() else 1.0
        log(f"step {step:6d} | loss {l:.4f} | lr {lr:.2e} | {tokens/1e6:8.1f}M tok | "
            f"{tps/1e3:5.1f}K tok/s | gnorm {gmax:6.2f} | scale {sc:.0f} | skip {skipped_win}/{skipped_total} | "
            f"left {(TRAIN_DEADLINE-now)/3600:.2f}h")
        csv(f"{step},{tokens},{l:.5f},{lr:.3e},{tps:.0f},{(now-run_t0)/3600:.3f},,,{gmax:.4f},{sc:.0f},{skipped_total}")
        win_t0, win_tok = now, 0
        gn_max_win.zero_(); skipped_win = 0


    flags = torch.tensor([
        float((now >= TRAIN_DEADLINE) or (total_steps is not None and step >= total_steps)),
        float(now - last_val >= TR.get("val_every_min", 30) * 60),
        float(now - last_ckpt >= TR.get("ckpt_every_min", 20) * 60)], device=DEV)
    if DDP_ON:
        dist.broadcast(flags, 0)
    stop, do_val, do_ckpt = (bool(f) for f in flags.tolist())

    if stop or do_val:
        v, per = evaluate()
        log(f"  [val] loss {v:.4f} ppl {math.exp(min(v, 20)):.2f} | " +
            " ".join(f"{k}={x:.3f}" for k, x in per.items()))
        csv(f"{step},{tokens},,{lr:.3e},,{(now-run_t0)/3600:.3f},{v:.5f},"
            + ";".join(f"{k}={x:.4f}" for k, x in per.items()) + ",,,")
        if v < best_val:
            best_val = v
            save_model_only("best", step, tokens, v)
            log(f"  new best -> {NAME}_best.pt")
        last_val = time.time()

    if stop or do_ckpt:
        save_training_ckpt(step, tokens, total_steps, best_val)
        last_ckpt = time.time()

    if stop:
        break

save_model_only("final", step, tokens, best_val)
log(f"Training finished: {step} steps, {tokens/1e6:.1f}M tokens, best val {best_val:.4f}")
if DDP_ON:
    dist.barrier()
    dist.destroy_process_group()

In [ ]:
%%writefile generate.py
import glob, json, os, re, sys, time, zipfile

import torch
import torch.distributed as dist
import torch.nn.functional as F

CFG = json.load(open(sys.argv[1]))
IN_PATH, OUT_PREFIX = sys.argv[2], sys.argv[3]
RUN, MODEL_CFG, GEN = CFG["run"], CFG["model"], CFG["gen"]
TARGET_WORDS = 128

if RUN.get("smoke_test"):
    import dummy_model as model_lib
else:
    import model as model_lib

if int(os.environ.get("WORLD_SIZE", "1")) > 1:
    LOCAL = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(LOCAL)
    try:
        dist.init_process_group("nccl", device_id=torch.device("cuda", LOCAL))
    except TypeError:
        dist.init_process_group("nccl")
    RANK, WORLD = dist.get_rank(), dist.get_world_size()
else:
    RANK, WORLD, LOCAL = 0, 1, 0
torch.cuda.set_device(LOCAL)
DEV = torch.device("cuda", LOCAL)
torch.manual_seed(int(GEN.get("seed", 0)) + RANK)


def log(*a):
    if RANK == 0:
        print(*a, flush=True)


# ----------------------------------------------------------------------------- tokenizer + model
from tokenizers import ByteLevelBPETokenizer

tok = ByteLevelBPETokenizer(vocab=os.path.join(RUN["tokenizer_dir"], "vocab.json"),
                            merges=os.path.join(RUN["tokenizer_dir"], "merges.txt"))
CTX = int(CFG["train"]["ctx_len"])
MODEL_CFG.setdefault("vocab_size", tok.get_vocab_size())
MODEL_CFG.setdefault("max_seq_len", CTX)

ck_choice = GEN["checkpoint"]
ck_path = ck_choice if os.path.exists(ck_choice) else os.path.join(RUN["out_dir"], f"{RUN['run_name']}_{ck_choice}.pt")
model = model_lib.build_model(MODEL_CFG).to(DEV)
ck = torch.load(ck_path, map_location=DEV, weights_only=False)
model.load_state_dict(ck["model"])
model.eval()
log(f"Loaded {ck_path} (step {ck.get('step')}, val {ck.get('val_loss')})")
del ck
HAS_CACHE = hasattr(model, "prefill") and hasattr(model, "decode_step")
log("Decoding with KV cache (prefill/decode_step)" if HAS_CACHE else
    "No prefill/decode_step on the model: recomputing the full sequence each step (slower)")

# ----------------------------------------------------------------------------- sampling utils
MAX_NEW = int(GEN["max_new_tokens"])
PROMPT_MAX = CTX - MAX_NEW
assert PROMPT_MAX > 64, "ctx_len too small for max_new_tokens"
BAN = [tok.token_to_id(t) for t in ("<s>", "</s>", "<unk>", "<pad>", "<mask>") if tok.token_to_id(t) is not None]


class NgramBlocker:

    def __init__(self, n, prompt_ids, scope):
        self.n, self.seen = n, {}
        self.hist = list(prompt_ids[-(n - 1):]) if (n and scope != "all") else []
        if n and scope == "all":
            for i in range(len(prompt_ids) - n + 1):
                self.seen.setdefault(tuple(prompt_ids[i:i + n - 1]), set()).add(prompt_ids[i + n - 1])
            self.hist = list(prompt_ids[-(n - 1):])
        self.scope = scope
        self.gen_count = 0

    def banned(self):
        if not self.n or len(self.hist) < self.n - 1:
            return ()
        return self.seen.get(tuple(self.hist[-(self.n - 1):]), ())

    def push(self, t):
        if self.n:
            key = tuple(self.hist[-(self.n - 1):])

            if len(key) == self.n - 1 and (self.scope == "all" or self.gen_count >= self.n - 1):
                self.seen.setdefault(key, set()).add(t)
            self.hist.append(t)
            self.gen_count += 1


def pick_next(logits, gen_tokens, blockers):
    if BAN:
        logits[:, BAN] = -float("inf")
    rp = GEN["rep_penalty"]
    if rp and rp != 1.0 and gen_tokens.numel():
        sc = torch.gather(logits, 1, gen_tokens)
        logits.scatter_(1, gen_tokens, torch.where(sc < 0, sc * rp, sc / rp))
    for b, bl in enumerate(blockers):
        bad = bl.banned()
        if bad:
            logits[b, list(bad)] = -float("inf")
    T = GEN["temperature"]
    if not T:
        return logits.argmax(-1)
    logits = logits / T
    k = GEN.get("top_k") or 0
    if k:
        kth = torch.topk(logits, k, dim=-1).values[:, -1:]
        logits = logits.masked_fill(logits < kth, -float("inf"))
    p = GEN.get("top_p") or 1.0
    probs = F.softmax(logits, -1)
    if p < 1.0:
        sp, si = probs.sort(-1, descending=True)
        sp[(sp.cumsum(-1) - sp) > p] = 0.0
        probs = torch.zeros_like(probs).scatter_(1, si, sp)
    probs = probs / probs.sum(-1, keepdim=True)
    return torch.multinomial(probs, 1).squeeze(1)


def clean(text):
    return re.sub(r"\s+", " ", text).strip()


@torch.inference_mode()
def generate_batch(prompts_ids):
    B, L = len(prompts_ids), len(prompts_ids[0])
    ids = torch.tensor(prompts_ids, dtype=torch.long, device=DEV)
    plen = torch.full((B,), L, dtype=torch.long, device=DEV)
    blockers = [NgramBlocker(GEN.get("no_repeat_ngram") or 0, p, GEN.get("ngram_scope", "generated"))
                for p in prompts_ids]
    gen = torch.empty((B, 0), dtype=torch.long, device=DEV)
    done = torch.zeros(B, dtype=torch.bool, device=DEV)
    with torch.autocast("cuda", dtype=torch.float16):
        if HAS_CACHE:
            logits, cache = model.prefill(ids, plen)
        else:
            logits = model(ids, plen)[:, -1]
    for step in range(MAX_NEW):
        nxt = pick_next(logits.float(), gen, blockers)
        gen = torch.cat([gen, nxt[:, None]], 1)
        nl = nxt.tolist()
        for b, bl in enumerate(blockers):
            bl.push(nl[b])
        if (step + 1) % 16 == 0:
            for b in range(B):
                if not done[b] and len(tok.decode(gen[b].tolist()).split()) > TARGET_WORDS:
                    done[b] = True
            if bool(done.all()):
                break
        if step == MAX_NEW - 1:
            break
        with torch.autocast("cuda", dtype=torch.float16):
            if HAS_CACHE:
                logits, cache = model.decode_step(nxt[:, None], cache)
            else:
                logits = model(torch.cat([ids, gen], 1), plen)[:, -1]
    return [clean(tok.decode(g.tolist())) for g in gen]


# ----------------------------------------------------------------------------- run
data = json.load(open(IN_PATH, encoding="utf-8"))
all_ids = list(data.keys())
if RUN.get("smoke_test"):
    all_ids = all_ids[:16]
mine = all_ids[RANK::WORLD]
enc = {k: tok.encode(data[k]["text"]).ids[-PROMPT_MAX:] for k in mine}
order = sorted(mine, key=lambda k: len(enc[k]))
BS = int(GEN["batch_size"])
out, dropped = {}, 0
t0 = time.time()
for i in range(0, len(order), BS):
    keys = order[i:i + BS]
    m = min(len(enc[k]) for k in keys)
    dropped += sum(len(enc[k]) - m for k in keys)
    texts = generate_batch([enc[k][-m:] for k in keys])
    for k, t in zip(keys, texts):
        out[k] = t
    if RANK == 0 and (i // BS) % 5 == 0:
        el = time.time() - t0
        print(f"  rank0 {i+len(keys)}/{len(order)} prompts | {el/60:.1f} min | "
              f"eta {el/(i+len(keys))*(len(order)-i-len(keys))/60:.1f} min", flush=True)


del model
torch.cuda.empty_cache()

part = f"{OUT_PREFIX}.part{RANK}.json"
json.dump(out, open(part, "w", encoding="utf-8"), ensure_ascii=False)
log(f"Context tokens trimmed for batching (rank0): {dropped} total")
if WORLD > 1:
    dist.barrier()

if RANK == 0:
    merged = {}
    for pf in sorted(glob.glob(f"{OUT_PREFIX}.part*.json")):
        merged.update(json.load(open(pf, encoding="utf-8")))
        os.remove(pf)
    preds, short = {}, 0
    for k in all_ids:
        w = merged.get(k, "").split()
        if len(w) < TARGET_WORDS:
            short += 1
        preds[k] = {"text": " ".join(w[:TARGET_WORDS])}
    pred_path = f"{OUT_PREFIX}_prediction.json"
    json.dump(preds, open(pred_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    zip_path = f"{OUT_PREFIX}_submission.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(pred_path, arcname="prediction.json")
    print(f"Wrote {pred_path} and {zip_path} | {len(preds)} ids | {short} under {TARGET_WORDS} words | "
          f"{(time.time()-t0)/60:.1f} min", flush=True)
if WORLD > 1:
    dist.destroy_process_group()

In [ ]:
# 5 — train (continuation)
_cap = int(SESSION_START + SESSION_HOURS * 3600 - RESERVE_MIN * 60 - time.time()) + 600
!timeout -k 60 {_cap} torchrun --standalone --nproc_per_node={NGPU} train.py run_config.json
_best = f"run/{NEW}_best.pt"
assert os.path.exists(_best), "TRAINING FAILED — see the output above; keep your existing submission"
NEW_VAL = torch.load(_best, map_location="cpu", weights_only=False).get("val_loss")
print(f"continued best val {NEW_VAL:.4f} | source best val {SRC_VAL}")
IMPROVED = SRC_VAL is None or NEW_VAL < SRC_VAL
print("IMPROVED -> generating" if IMPROVED else "NOT improved -> no zip; your existing submission stands")

In [ ]:
# 6 — decoding chosen on local BERTScore, then test set -> /kaggle/working/submission.zip
if IMPROVED:
    def rouge_l_f1(a, b):
        a, b = a.split(), b.split()
        if not a or not b: return 0.0
        prev = [0] * (len(b) + 1)
        for x in a:
            cur = [0]
            for j, y in enumerate(b):
                cur.append(prev[j] + 1 if x == y else max(prev[j + 1], cur[-1]))
            prev = cur
        l = prev[-1]
        if l == 0: return 0.0
        p, r = l / len(a), l / len(b)
        return 2 * p * r / (p + r)
    from bert_score import score as bs_score
    ref = json.load(open(f"{TOKEN_DIR}/local_ref.json")); KS = list(ref)
    results = {}
    for name, over in CANDS.items():
        if SESSION_START + SESSION_HOURS * 3600 - time.time() < 8 * 60 and results:
            print("time is short: stopping the sweep"); break
        c = copy.deepcopy(CFG); c["gen"].update(over); json.dump(c, open(f"gen_{name}.json", "w"))
        !timeout -k 30 900 torchrun --standalone --nproc_per_node={NGPU} generate.py gen_{name}.json {TOKEN_DIR}/local_test.json run/local_{name}
        pp = f"run/local_{name}_prediction.json"
        if not os.path.exists(pp): print(name, "failed"); continue
        p = json.load(open(pp))
        _, _, f = bs_score([p[k]["text"] for k in KS], [ref[k]["text"] for k in KS], lang="en", batch_size=32, verbose=False)
        torch.cuda.empty_cache()
        results[name] = dict(bertscore=float(f.mean()), rougeL=sum(rouge_l_f1(p[k]["text"], ref[k]["text"]) for k in KS) / len(KS), gen=over)
        print(f"  {name:18s} BERTScore {results[name]['bertscore']:.4f} | ROUGE-L {results[name]['rougeL']:.4f}")
    json.dump(results, open(f"run/{NEW}_decoding_local.json", "w"), indent=1)
    BEST = max(results, key=lambda k: results[k]["bertscore"])
    print("test generation will use:", BEST)
    !timeout -k 30 1500 torchrun --standalone --nproc_per_node={NGPU} generate.py gen_{BEST}.json {TEST_PATH} run/test
    assert os.path.exists("run/test_submission.zip"), "TEST GENERATION FAILED — keep your existing submission"
    shutil.copy("run/test_submission.zip", "/kaggle/working/submission.zip")
    import zipfile
    test = json.load(open(TEST_PATH, encoding="utf-8"))
    with zipfile.ZipFile("/kaggle/working/submission.zip") as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
        pred = json.loads(z.read("prediction.json").decode("utf-8"))
    assert set(pred) == set(test), "id mismatch"
    wc = [len(v["text"].split()) for v in pred.values()]
    print(f"OK: {len(pred)} ids | words min/mean/max {min(wc)}/{sum(wc)/len(wc):.1f}/{max(wc)} | empty {sum(w == 0 for w in wc)}")
    for f in glob.glob("run/local_*_submission.zip"):
        os.remove(f)
    print(f"submission.zip = {NEW} (val {NEW_VAL:.4f} vs source {SRC_VAL}) with decoding {BEST}")